# Amazon Business Entity Resolution: High-Recall Blocking Optimization
### Objective: Maximize Blocking Recall (Target >=99%) on 50k Pilot Benchmark
- Audit 13.31% missed matches from baseline across 18 specific failure modes.
- Evaluate individual blocker contributions and unique value.
- Progressively evaluate Versions 1 through 8 (Char N-Grams, Rare Token Pairs, Address Signatures, Phonetic Retrieval, Uncapped Indexes).
- Validate frozen blocker on independent 50k Pilot B.
- Save checkpoints and analytical reports.

In [ ]:
import os
import sys
import shutil
import psutil
from pathlib import Path

print('=== ENVIRONMENT & RESOURCE CHECK ===')
ram = psutil.virtual_memory()
print(f'Total RAM: {ram.total / (1024**3):.2f} GB | Available: {ram.available / (1024**3):.2f} GB')
print(f'CPU Count: {os.cpu_count()}')

# Setup working directory structure
BASE_WORKING = Path('/kaggle/working/business_er') if Path('/kaggle/working').exists() else Path('./business_er')
for sub in ['src', 'checkpoints/blocking', 'cache', 'reports', 'output', 'logs']:
    (BASE_WORKING / sub).mkdir(parents=True, exist_ok=True)

# Clone repository to obtain latest modular src codebase
REPO_DIR = Path('/kaggle/working/repo')
if not (REPO_DIR / 'src').exists():
    print('Cloning project repository from GitHub...')
    ret = os.system('git clone --depth 1 https://github.com/bpsd07/ML-challenge.git /kaggle/working/repo')
    print(f'Git clone exited with: {ret}')

if (REPO_DIR / 'src').exists():
    for f in (REPO_DIR / 'src').glob('*.py'):
        shutil.copy(f, BASE_WORKING / 'src' / f.name)
    print(f'Successfully synced src modules from {REPO_DIR / "src"} to {BASE_WORKING / "src"}')
else:
    print('Fallback: checking local src directory')

sys.path.insert(0, str(BASE_WORKING / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, 'src')


In [ ]:
print('=== EXECUTING HIGH-RECALL BLOCKING AUDIT & OPTIMIZATION ===')
import evaluate_blocking_pipeline

# Run the complete blocking evaluation and audit pipeline
evaluate_blocking_pipeline.main()


In [ ]:
print('=== COPYING REPORTS & CHECKPOINTS TO WORKING ROOT FOR EXPORT ===')
export_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

reports_src = BASE_WORKING / 'reports'
checkpoints_src = BASE_WORKING / 'checkpoints'

if reports_src.exists():
    dest_rep = export_dir / 'reports'
    dest_rep.mkdir(parents=True, exist_ok=True)
    for f in reports_src.glob('*'):
        if f.is_file():
            shutil.copy(f, dest_rep / f.name)
            print(f'Exported report: {f.name} ({f.stat().st_size / 1024:.1f} KB)')

if checkpoints_src.exists():
    dest_chk = export_dir / 'checkpoints'
    dest_chk.mkdir(parents=True, exist_ok=True)
    for f in checkpoints_src.rglob('*.json'):
        rel = f.relative_to(checkpoints_src)
        out_target = dest_chk / rel
        out_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, out_target)
        print(f'Exported checkpoint: {rel}')

print('\n=== ALL ARTIFACTS READY FOR DOWNLOAD VIA KAGGLE CLI ===')
